# Part 2 — Data Prep & Claims Development (Emerging Market)

Replicates the v4 developed-market pipeline with adaptations for the emerging-market dataset.

**Key adaptations vs v4:**
- Age bands mapped to numeric midpoints (emerging uses categorical `_age_band` instead of `_age`)
- BF development factors borrowed from Part 1 (emerging has only 2023-2024, no mature years to derive factors)
- `gender` available as a new feature (not in developed data)
- Motor missing `annual_mileage`, `telematics_opt_in`, `telematics_score` — dropped automatically
- Only 2 years of data (2023-2024) vs 4 years (2021-2024) in developed

In [0]:
# Step 1: Import & load all emerging parquet files
import pandas as pd
import numpy as np
import os
from pathlib import Path

DATA_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data_emerging/'

files_to_load = [
    'policies_motor', 'policies_buildings', 'policies_contents',
    'test_motor', 'test_buildings', 'test_contents',
    'claims', 'customers', 'addresses', 'weather', 'sample_submission'
]

dfs = {}
for name in files_to_load:
    path = os.path.join(DATA_DIR, f'{name}.parquet')
    dfs[name] = pd.read_parquet(path)
    print(f'{name}: {dfs[name].shape}')

# Also load developed data for BF factor reference and target encoding priors
DEV_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data/'
dfs_dev_claims = pd.read_parquet(os.path.join(DEV_DIR, 'claims.parquet'))
print(f"\nDeveloped claims loaded for reference: {dfs_dev_claims.shape}")

In [0]:
# Step 2: Data cleaning

# 2a. Fix exposure = (period_end - period_start + 1) / 365
for name in ['policies_motor', 'policies_buildings', 'policies_contents',
             'test_motor', 'test_buildings', 'test_contents']:
    df = dfs[name]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    old_exp = df['exposure'].copy()
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    print(f'{name}: exposure fixed, mean={df["exposure"].mean():.4f} (was {old_exp.mean():.4f})')

# 2b. Create Year field from period_start
for name in ['policies_motor', 'policies_buildings', 'policies_contents',
             'test_motor', 'test_buildings', 'test_contents']:
    dfs[name]['Year'] = pd.to_datetime(dfs[name]['period_start']).dt.year
    print(f'{name}: Year range {dfs[name]["Year"].min()}-{dfs[name]["Year"].max()}')

# 2c. Map age_band columns to numeric midpoints
AGE_BAND_MAP = {
    '17-24': 20.5,
    '25-34': 29.5,
    '35-49': 42.0,
    '50-64': 57.0,
    '65+': 70.0,
}
# Find all _age_band columns and map them
for name, df in dfs.items():
    for col in df.columns:
        if col.endswith('_age_band') or col == 'age_band':
            new_col = col.replace('_age_band', '_age')
            df[new_col] = df[col].map(AGE_BAND_MAP)
            print(f'{name}: mapped {col} -> {new_col} (midpoints), values: {df[new_col].value_counts().to_dict()}')

# 2d. construction_type: normalise uppercase, combine W and T as TIMBER
for name in ['policies_buildings', 'test_buildings']:
    df = dfs[name]
    if 'construction_type' in df.columns:
        df['construction_type'] = df['construction_type'].astype(str).str.strip().str.upper()
        df['construction_type'] = df['construction_type'].replace({'W': 'TIMBER', 'T': 'TIMBER'})
        print(f'{name}: construction_type = {df["construction_type"].value_counts().to_dict()}')

# 2e. Strip whitespace from all ID columns
id_cols = ['policy_id', 'customer_id', 'address_id', 'claim_id', 'row_id']
for name, df in dfs.items():
    for col in id_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

# 2f. Check for full row duplicates in policy files
for name in ['policies_motor', 'policies_buildings', 'policies_contents']:
    df = dfs[name]
    n_before = len(df)
    df.drop_duplicates(inplace=True)
    n_after = len(df)
    print(f'{name}: {n_before} rows -> {n_after} rows (dropped {n_before - n_after} duplicates)')

# 2g. Check compulsory_excess - drop if constant per line
for name in ['policies_motor', 'policies_buildings', 'policies_contents']:
    df = dfs[name]
    if 'compulsory_excess' in df.columns:
        n_unique = df['compulsory_excess'].nunique()
        print(f'{name}: compulsory_excess unique values = {n_unique}')
        if n_unique <= 1:
            df.drop('compulsory_excess', axis=1, inplace=True)
            print(f'  -> Dropped (constant)')

# 2h. Zero-pad customer_id if lengths vary (check but may not be needed for emerging)
for name in ['policies_motor', 'test_motor']:
    df = dfs[name]
    if 'customer_id' in df.columns:
        lens = df['customer_id'].str.len().value_counts()
        if lens.nunique() > 1:
            print(f'{name}: customer_id lengths vary: {lens.to_dict()} -> zero-padding to max')
            max_len = lens.index.max()
            df['customer_id'] = df['customer_id'].str.zfill(max_len)
        else:
            print(f'{name}: customer_id all {lens.index[0]} chars')

# Also check customers file
cust_lens = dfs['customers']['customer_id'].str.len().value_counts()
if cust_lens.nunique() > 1:
    max_len = cust_lens.index.max()
    dfs['customers']['customer_id'] = dfs['customers']['customer_id'].str.zfill(max_len)
    print(f'customers: zero-padded customer_id to {max_len} chars')
else:
    print(f'customers: customer_id all {cust_lens.index[0]} chars')

# 2i. Check sum_insured for potential x1000 errors in buildings/contents
for name in ['policies_buildings', 'policies_contents']:
    df = dfs[name]
    if 'sum_insured' in df.columns:
        med_si = df['sum_insured'].median()
        high_mask = df['sum_insured'] > 100 * med_si
        n_high = high_mask.sum()
        print(f'{name}: sum_insured median={med_si:.0f}, >100x median: {n_high}')
        if n_high > 0 and n_high < len(df) * 0.01:
            df.loc[high_mask, 'sum_insured'] = df.loc[high_mask, 'sum_insured'] / 1000
            print(f'  -> Fixed {n_high} values by dividing by 1000')
        elif n_high > 0:
            print(f'  -> Kept as-is ({n_high} values >100x median, but >1% of data — likely genuine)')

In [0]:
# Step 3: Join enrichment data to policy files
# Customers: left join on customer_id (drop PII columns if present)
# Addresses: left join on address_id
# Weather: left join on address_id + Year (one row per address-year)

customers_clean = dfs['customers'].drop(columns=['surname', 'date_of_birth'], errors='ignore')
addresses_clean = dfs['addresses'].copy()

# Weather: aggregate to address_id + year level
weather = dfs['weather'].copy()
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)
print(f'Weather: {len(weather)} rows -> {len(weather_agg)} rows after aggregation')
print(f'Weather Year range: {weather_agg["Year"].min()}-{weather_agg["Year"].max()}')

policy_files = {
    'motor': 'policies_motor',
    'buildings': 'policies_buildings',
    'contents': 'policies_contents'
}

for line, fname in policy_files.items():
    df = dfs[fname]
    n_before = len(df)
    cols_before = df.shape[1]

    # Join customers
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    n_after_cust = len(df)

    # Join addresses
    df = df.merge(addresses_clean, on='address_id', how='left', suffixes=('', '_addr'))
    n_after_addr = len(df)

    # Join weather on address_id + Year
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    n_after_weather = len(df)

    dfs[fname] = df
    print(f'\n{line}: {n_before} rows -> {len(df)} rows, {cols_before} -> {df.shape[1]} cols')
    print(f'  After customers: {n_after_cust} rows, after addresses: {n_after_addr} rows, after weather: {n_after_weather} rows')

    # Check join quality
    for ref_name, ref_df in [('customers', customers_clean), ('addresses', addresses_clean), ('weather', weather_agg)]:
        join_col = 'customer_id' if ref_name == 'customers' else 'address_id'
        matched = df[join_col].isin(ref_df[join_col]).sum() if join_col in df.columns else 0
        print(f'  {ref_name}: {matched}/{len(df)} matched ({100*matched/len(df):.1f}%)')

    # Show null counts for key enrichment columns
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0].sort_values(ascending=False).head(10)
    if len(nulls) > 0:
        print(f'  Top null columns: {nulls.to_dict()}')

In [0]:
# Step 4: Claims processing

claims = dfs['claims']

# 4a. Create incurred_net_of_recoveries = incurred_to_date - recoveries
claims['incurred_net_of_recoveries'] = claims['incurred_to_date'] - claims['recoveries']

# 4b. Check status values and exclude declined/repudiated
print('=== Claims status ===')
print(claims['status'].value_counts().to_dict())
declined_statuses = ['declined', 'repudiated']
declined_mask = claims['status'].str.lower().isin(declined_statuses)
n_declined = declined_mask.sum()
print(f'Declined/repudiated claims: {n_declined} (excluded from target)')

claims_active = claims[~declined_mask].copy()
print(f'Active claims: {len(claims_active)}')

# 4c. Explore perils
print(f'\n=== Perils ===')
print(claims_active.groupby(['line', 'peril']).agg(
    count=('claim_id', 'count'),
    mean_incurred=('incurred_net_of_recoveries', 'mean'),
    median_incurred=('incurred_net_of_recoveries', 'median'),
    sum_incurred=('incurred_net_of_recoveries', 'sum')
).to_string())

# 4d. Check for negative incurred_net_of_recoveries
n_neg = (claims_active['incurred_net_of_recoveries'] < 0).sum()
print(f'\nNegative incurred_net_of_recoveries: {n_neg}')
if n_neg > 0:
    print('  (salvage/recovery exceeds incurred — kept as-is, represents net gain)')

# 4e. Summary stats
print(f'\n=== incurred_net_of_recoveries stats ===')
print(claims_active['incurred_net_of_recoveries'].describe())

# Store active claims
dfs['claims_active'] = claims_active

In [0]:
# Step 5: Claims development to ultimate using Bornhuetter-Ferguson (BF) method
# KEY ADAPTATION: Emerging has only 2023-2024 (no mature years to derive factors from).
# We use Part 1's per-peril BF factors as a starting point — same insurance product,
# different market but similar reporting patterns are the best available assumption.
#
# Part 1 factors were derived from chain-ladder on 2021-2022 mature years:
#   Motor: BI=1.82, TPP=1.08, short_motor=1.08 for 2024
#   Buildings: subsidence=1.47, buildings_other=1.08 for 2024
#   Contents: contents=1.04 for 2024
#   2023 factors are smaller (more developed): BI=1.06, TPP=1.01, etc.

claims_active = dfs['claims_active']

# 5a. Build reported claim COUNT triangle by accident year x development year
claims_active['loss_year'] = pd.to_datetime(claims_active['loss_date']).dt.year
claims_active['report_date_dt'] = pd.to_datetime(claims_active['report_date'], errors='coerce')
claims_active['report_year'] = claims_active['report_date_dt'].dt.year
claims_active['dev_lag'] = claims_active['report_year'] - claims_active['loss_year']

# Claim count triangle (cumulative)
triangle_raw = claims_active.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
count_triangle = triangle_raw.cumsum(axis=1)
print('=== Reported Claim Count Triangle (cumulative) ===')
print(count_triangle)
print(f'\nNote: Emerging has only 2023-2024 — no mature years to derive factors from')

# 5b. Development factors — borrowed from Part 1 (developed market)
# These are the SAME factors used in v4, applied to the corresponding accident years.
# 2023 in emerging uses the 2023 factors from Part 1 (moderately mature, ~1 year of development)
# 2024 in emerging uses the 2024 factors from Part 1 (least mature, ~0 years of development)
DEV_FACTORS = {
    2023: {'bi': 1.06, 'tpp': 1.01, 'short_motor': 1.01, 'subsidence': 1.15, 'buildings_other': 1.02, 'contents': 1.01},
    2024: {'bi': 1.82, 'tpp': 1.08, 'short_motor': 1.08, 'subsidence': 1.47, 'buildings_other': 1.08, 'contents': 1.04},
}

LONG_TAIL_PERILS = ['bodily_injury', 'third_party_property', 'third_party_injury']

def get_dev_factor(peril, line, loss_year):
    """Get development factor based on peril type, line, and accident year."""
    peril_lower = str(peril).lower()
    if line == 'motor':
        if 'bodily' in peril_lower or 'injury' in peril_lower:
            tail = 'bi'
        elif 'third_party' in peril_lower:
            tail = 'tpp'
        else:
            tail = 'short_motor'
    elif line == 'buildings':
        if 'subsidence' in peril_lower:
            tail = 'subsidence'
        else:
            tail = 'buildings_other'
    else:  # contents
        tail = 'contents'
    return DEV_FACTORS.get(loss_year, {}).get(tail, 1.0)

# 5c. Apply development factors
claims_active['dev_factor'] = claims_active.apply(
    lambda r: get_dev_factor(r['peril'], r['line'], r['loss_year']), axis=1
)
claims_active['ultimate_claims'] = claims_active['incurred_net_of_recoveries'] * claims_active['dev_factor']
claims_active['ultimate_no_dev'] = claims_active['incurred_net_of_recoveries']

print('\n=== Development factors by line, peril, accident year ===')
print(claims_active.groupby(['line', 'peril', 'loss_year', 'dev_factor']).agg(
    count=('claim_id', 'count'),
    reported_incurred=('incurred_net_of_recoveries', 'sum'),
    ultimate=('ultimate_claims', 'sum')
).to_string())

# 5d. Summary by peril and accident year
summary_by_peril = claims_active.groupby(['line', 'peril', 'loss_year']).agg(
    claim_count=('claim_id', 'count'),
    reported_incurred=('incurred_net_of_recoveries', 'sum'),
    dev_factor=('dev_factor', 'first'),
    ultimate=('ultimate_claims', 'sum'),
).reset_index()
summary_by_peril['avg_severity'] = summary_by_peril['ultimate'] / summary_by_peril['claim_count']
print('\n=== Summary by peril and accident year ===')
print(summary_by_peril.to_string())

# Store
dfs['claims_active'] = claims_active
claims_dev_summary = summary_by_peril
print(f'\nTotal reported incurred (active): {claims_active["incurred_net_of_recoveries"].sum():.2f}')
print(f'Total ultimate claims (developed): {claims_active["ultimate_claims"].sum():.2f}')
print(f'Development uplift: {(claims_active["ultimate_claims"].sum() / claims_active["incurred_net_of_recoveries"].sum() - 1)*100:.1f}%')

In [0]:
# Step 5b: Match claims to policy rows and aggregate ultimate_claims by row_id
# Match: policy_id + year(loss_date) == year(period_start) + loss_date within [period_start, period_end]

claims_active = dfs['claims_active']
claims_active['loss_date_dt'] = pd.to_datetime(claims_active['loss_date'])

for line, fname in policy_files.items():
    df = dfs[fname].copy()
    df['_ps_dt'] = pd.to_datetime(df['period_start'])
    df['_pe_dt'] = pd.to_datetime(df['period_end'])

    line_claims = claims_active[claims_active['line'] == line].copy()
    print(f'\n=== {line}: {len(line_claims)} claims -> {len(df)} policy rows ===')

    # Merge on policy_id
    merged = line_claims.merge(
        df[['row_id', 'policy_id', '_ps_dt', '_pe_dt', 'Year']],
        on='policy_id', how='inner'
    )

    # Filter: loss_year == Year AND loss_date within [period_start, period_end]
    merged = merged[merged['loss_date_dt'].dt.year == merged['Year']]
    mask = (merged['loss_date_dt'] >= merged['_ps_dt']) & (merged['loss_date_dt'] <= merged['_pe_dt'])
    matched = merged[mask].copy()

    # If multiple rows match a claim, keep the first (sorted by row_id)
    matched = matched.sort_values('row_id').drop_duplicates('claim_id', keep='first')

    n_matched = len(matched)
    n_unmatched = len(line_claims) - n_matched
    print(f'  Matched: {n_matched} claims ({n_unmatched} unmatched)')

    # Aggregate by row_id
    agg = matched.groupby('row_id').agg(
        ultimate_claims=('ultimate_claims', 'sum'),
        claim_count=('claim_id', 'count'),
        reported_incurred=('incurred_net_of_recoveries', 'sum'),
    ).reset_index()

    # Join back to policy data
    df = df.merge(agg, on='row_id', how='left')
    df['ultimate_claims'] = df['ultimate_claims'].fillna(0)
    df['claim_count'] = df['claim_count'].fillna(0).astype(int)
    df['reported_incurred'] = df['reported_incurred'].fillna(0)

    # Clean up temp columns
    df.drop(['_ps_dt', '_pe_dt'], axis=1, inplace=True)

    dfs[fname] = df
    n_with_claims = (df['claim_count'] > 0).sum()
    print(f'  Rows with claims: {n_with_claims} ({100*n_with_claims/len(df):.1f}%)')
    print(f'  Total ultimate_claims: {df["ultimate_claims"].sum():,.2f}')
    print(f'  Mean ultimate_claims (all rows): {df["ultimate_claims"].mean():.4f}')
    print(f'  Mean ultimate_claims (rows with claims): {df.loc[df["claim_count"]>0, "ultimate_claims"].mean():.2f}')

In [0]:
# Step 6: Trend analysis 2023-2024
# With only 2 years, we cannot reliably estimate trend.
# We use Part 1's trend factors as a guide (same insurance product).
# These will be applied to 2025 test predictions in the modeling notebooks.

print('=== Trend Analysis 2023-2024 (using developed ultimate_claims) ===\n')

trend_data = {}
trend_factors = {}
for line in ['motor', 'buildings', 'contents']:
    df = dfs[policy_files[line]]
    by_year = df.groupby('Year').agg(
        exposure=('exposure', 'sum'),
        claims=('ultimate_claims', 'sum'),
        claim_count=('claim_count', 'sum'),
        policies=('row_id', 'count'),
    )
    by_year['claims_rate'] = by_year['claims'] / by_year['exposure']
    by_year['frequency'] = by_year['claim_count'] / by_year['exposure']
    print(f'--- {line.upper()} ---')
    print(by_year.to_string())

    # Calculate year-on-year change
    if len(by_year) >= 2:
        rates = by_year['claims_rate'].values
        pct_change = (rates[-1] / rates[0] - 1) * 100
        print(f'  2023->2024 claims_rate change: {pct_change:+.1f}%')
        print(f'  NOTE: Only 2 years — trend estimate is unreliable')
        print(f'  Using Part 1 trend factor as guide')
    trend_data[line] = by_year
    print()

# Part 1 trend factors (from v4 — same insurance product, more data to estimate)
TREND_FACTORS = {
    'motor': 1.04,
    'buildings': 1.096,
    'contents': 1.049,
}
print(f'Trend uplift factors for 2025 (borrowed from Part 1): {TREND_FACTORS}')

In [0]:
# Step 7: Save processed data as parquet to data_emerging/proc/
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl", "-q"])

PROC_DIR = os.path.join(DATA_DIR, 'proc')
os.makedirs(PROC_DIR, exist_ok=True)

EXCEL_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/Excel files_emerging_v1/'
os.makedirs(EXCEL_DIR, exist_ok=True)

for line, fname in policy_files.items():
    df = dfs[fname]
    out_path = os.path.join(PROC_DIR, f'proc_{line}.parquet')
    df.to_parquet(out_path, index=False)
    print(f'Saved: {out_path} ({len(df)} rows, {df.shape[1]} cols)')
    print(f'  Target: ultimate_claims (mean={df["ultimate_claims"].mean():.4f}, sum={df["ultimate_claims"].sum():,.2f})')
    print(f'  Rows with claims: {(df["claim_count"]>0).sum()} ({100*(df["claim_count"]>0).mean():.1f}%)')

# Excel checkpoint
with pd.ExcelWriter(os.path.join(EXCEL_DIR, 'claims_development_summary.xlsx'), engine='openpyxl') as writer:
    claims_dev_summary.to_excel(writer, sheet_name='Dev Factors', index=False)
    count_triangle.to_excel(writer, sheet_name='Claim Count Triangle')
    trend_all = pd.concat([trend_data[line].reset_index().assign(line=line) for line in trend_data], ignore_index=True)
    trend_all.to_excel(writer, sheet_name='Trend Analysis', index=False)
    assumptions = pd.DataFrame({
        'Assumption': [
            'BF development factors borrowed from Part 1 (developed market)',
            'Motor: BI=1.82 for 2024, TPP=1.08, short_motor=1.08',
            'Buildings: subsidence=1.47 for 2024, buildings_other=1.08',
            'Contents: 1.04 for 2024',
            '2023 factors smaller (more developed): BI=1.06, subsidence=1.15, etc.',
            'Emerging has only 2023-2024 — no mature years to validate factors',
            'Trend factors borrowed from Part 1: motor 1.04, buildings 1.096, contents 1.049',
            'Age bands mapped to midpoints: 17-24=20.5, 25-34=29.5, 35-49=42, 50-64=57, 65+=70',
            'Declined/repudiated claims excluded from target',
            'compulsory_excess dropped if constant per line',
        ],
        'Source': [
            'Part 1 v4 (chain-ladder derived)', 'Part 1 v4', 'Part 1 v4', 'Part 1 v4',
            'Part 1 v4', 'AI (Genie Code)',
            'Part 1 v4', 'AI (Genie Code)',
            'AI (Genie Code)', 'AI (Genie Code)',
        ]
    })
    assumptions.to_excel(writer, sheet_name='Assumptions', index=False)

print(f'\nExcel checkpoint saved: {EXCEL_DIR}claims_development_summary.xlsx')
print(f'\nProcessed data saved to: {PROC_DIR}')

## Step 8: AI/Human Decision Log

### What AI did
- Imported all 13 emerging parquet files + developed claims for BF factor reference
- Data cleaning: exposure recalculation, Year field, construction_type normalisation, age band mapping, ID stripping, duplicate check, compulsory_excess drop, sum_insured check, customer_id padding check
- Joined customers + addresses + weather to all 3 policy files
- Claims processing: incurred_net_of_recoveries, excluded repudiated claims
- Applied BF development factors borrowed from Part 1 (per-peril by accident year)
- Matched claims to policy rows (policy_id + loss_date within period)
- Trend analysis (2 years only — unreliable, using Part 1 factors as guide)
- Saved processed parquet files and Excel checkpoint

### Key adaptations vs v4 (developed market)
1. **BF factors borrowed from Part 1** — emerging has only 2023-2024, no mature years to derive factors
2. **Age bands mapped to numeric midpoints** — emerging uses categorical `_age_band` instead of numeric `_age`
3. **`gender` available** — new feature not in developed data
4. **Motor missing** `annual_mileage`, `telematics_opt_in`, `telematics_score` — dropped automatically by feature selection
5. **Trend factors from Part 1** — only 2 years of data, cannot estimate trend reliably

### Key assumptions
1. BF development factors from Part 1 apply to emerging market (same insurance product, different market)
2. Age band midpoints are reasonable proxies for numeric age
3. Trend factors from Part 1 are a reasonable guide for 2025 predictions
4. Claims matching logic is the same as developed (policy_id + loss_date within period)